##Import pyspark Libraries and create Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [2]:
spark = SparkSession.builder.appName("Data Analysis").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/24 11:55:35 WARN Utils: Your hostname, Aayushs-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 10.1.2.171 instead (on interface en0)
26/05/24 11:55:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 11:55:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/24 11:55:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Load / Import DATA

In [48]:
df  = spark.read.csv("employees.csv", header = True, inferSchema=True)
'''
schema = StructType ([
StructField("id", IntegerType(), nullable = False)
'''

#json = spar.read.json(), spark.read.parquet()

'\nschema = StructType ([\nStructField("id", IntegerType(), nullable = False)\n'

In [4]:
print(df)

DataFrame[id: int, name: string, department: string, salary: int, age: int]


In [5]:
df.show()

+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|  NULL|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|       NULL| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [6]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [7]:
df.dtypes

[('id', 'int'),
 ('name', 'string'),
 ('department', 'string'),
 ('salary', 'int'),
 ('age', 'int')]

Check For Any Null Values

In [8]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

In [9]:
null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

In [10]:
null_counts.show()


+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  0|   0|         1|     1|  1|
+---+----+----------+------+---+



In [11]:
df.filter(col("name").isNull()).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
+---+----+----------+------+---+



In [12]:
df_clean = df.dropna()
df_clean.show()

+---+-----+-----------+------+---+
| id| name| department|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 85000| 29|
|  2|  Bob|  Marketing| 62000| 34|
|  3|Carol|Engineering| 91000| 41|
|  5|  Eve|  Marketing| 70000| 38|
|  7|Grace|Engineering| 95000| 35|
|  8|Henry|         HR| 58000| 45|
|  9|  Ivy|  Marketing| 67000| 31|
| 10| Jack|Engineering| 78000| 28|
+---+-----+-----------+------+---+



In [13]:
df_filled = df.fillna({"salary":0, "department":"Unknown"})
df_filled.show()

+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|     0|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|    Unknown| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [14]:
df.describe().show()

26/05/24 11:55:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----+-----------+-----------------+-----------------+
|summary|                id| name| department|           salary|              age|
+-------+------------------+-----+-----------+-----------------+-----------------+
|  count|                10|   10|          9|                9|                9|
|   mean|               5.5| NULL|       NULL|73333.33333333333|34.22222222222222|
| stddev|3.0276503540974917| NULL|       NULL|14696.93845669907|6.180165405913052|
|    min|                 1|Alice|Engineering|            54000|               27|
|    max|                10| Jack|  Marketing|            95000|               45|
+-------+------------------+-----+-----------+-----------------+-----------------+



In [15]:
df.describe("salary").show()

+-------+-----------------+
|summary|           salary|
+-------+-----------------+
|  count|                9|
|   mean|73333.33333333333|
| stddev|14696.93845669907|
|    min|            54000|
|    max|            95000|
+-------+-----------------+



Selecting Columns

In [16]:
df.select("name").show()

+-----+
| name|
+-----+
|Alice|
|  Bob|
|Carol|
|David|
|  Eve|
|Frank|
|Grace|
|Henry|
|  Ivy|
| Jack|
+-----+



In [17]:
df.select(col("name"), col("salary") , col("department")).show()

+-----+------+-----------+
| name|salary| department|
+-----+------+-----------+
|Alice| 85000|Engineering|
|  Bob| 62000|  Marketing|
|Carol| 91000|Engineering|
|David|  NULL|         HR|
|  Eve| 70000|  Marketing|
|Frank| 54000|       NULL|
|Grace| 95000|Engineering|
|Henry| 58000|         HR|
|  Ivy| 67000|  Marketing|
| Jack| 78000|Engineering|
+-----+------+-----------+



In [18]:
df.select(
    col("name"),
    col("salary"),(col("salary") * 2).alias("Salary_with_bonus")).show()

+-----+------+-----------------+
| name|salary|Salary_with_bonus|
+-----+------+-----------------+
|Alice| 85000|           170000|
|  Bob| 62000|           124000|
|Carol| 91000|           182000|
|David|  NULL|             NULL|
|  Eve| 70000|           140000|
|Frank| 54000|           108000|
|Grace| 95000|           190000|
|Henry| 58000|           116000|
|  Ivy| 67000|           134000|
| Jack| 78000|           156000|
+-----+------+-----------------+



In [19]:
df = df.withColumn("salary_band", when(col("salary") >= 80000, "High")
                   .when(col("salary") >= 60000, "Mid")
                   .otherwise("Low"))
df.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
| 10| Jack|Engineering| 78000|  28|        Mid|
+---+-----+-----------+------+----+-----------+



Rows Selection

In [20]:
df.filter(col("salary") > 65000).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [22]:
df.filter(col("department") == "Engineering").show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [23]:
df.where(col("department") == "Engineering").show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [24]:
df.where(col("salary") > 65000).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [30]:
df.where((col("salary") > 60000 ) & (col("age") < 40)).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [31]:
df.limit(3).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  3|Carol|Engineering| 91000| 41|       High|
+---+-----+-----------+------+---+-----------+



In [32]:
df_1 = df

In [33]:
df_1.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
| 10| Jack|Engineering| 78000|  28|        Mid|
+---+-----+-----------+------+----+-----------+



Delete column

In [34]:
df_1.drop("age").show()

+---+-----+-----------+------+-----------+
| id| name| department|salary|salary_band|
+---+-----+-----------+------+-----------+
|  1|Alice|Engineering| 85000|       High|
|  2|  Bob|  Marketing| 62000|        Mid|
|  3|Carol|Engineering| 91000|       High|
|  4|David|         HR|  NULL|        Low|
|  5|  Eve|  Marketing| 70000|        Mid|
|  6|Frank|       NULL| 54000|        Low|
|  7|Grace|Engineering| 95000|       High|
|  8|Henry|         HR| 58000|        Low|
|  9|  Ivy|  Marketing| 67000|        Mid|
| 10| Jack|Engineering| 78000|        Mid|
+---+-----+-----------+------+-----------+



In [38]:
df_1.filter((col("department") != "HR") & (col("department") != "Marketing") ).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [39]:
df_1.withColumnRenamed("salary", "annual_salary").show()

+---+-----+-----------+-------------+----+-----------+
| id| name| department|annual_salary| age|salary_band|
+---+-----+-----------+-------------+----+-----------+
|  1|Alice|Engineering|        85000|  29|       High|
|  2|  Bob|  Marketing|        62000|  34|        Mid|
|  3|Carol|Engineering|        91000|  41|       High|
|  4|David|         HR|         NULL|  27|        Low|
|  5|  Eve|  Marketing|        70000|  38|        Mid|
|  6|Frank|       NULL|        54000|NULL|        Low|
|  7|Grace|Engineering|        95000|  35|       High|
|  8|Henry|         HR|        58000|  45|        Low|
|  9|  Ivy|  Marketing|        67000|  31|        Mid|
| 10| Jack|Engineering|        78000|  28|        Mid|
+---+-----+-----------+-------------+----+-----------+



Change the Column Names of Entire Table

In [41]:
new_names = ["emp_id", "full_name", "dept","pay","years", "s_band"]
df_1.toDF(*new_names).show()

+------+---------+-----------+-----+-----+------+
|emp_id|full_name|       dept|  pay|years|s_band|
+------+---------+-----------+-----+-----+------+
|     1|    Alice|Engineering|85000|   29|  High|
|     2|      Bob|  Marketing|62000|   34|   Mid|
|     3|    Carol|Engineering|91000|   41|  High|
|     4|    David|         HR| NULL|   27|   Low|
|     5|      Eve|  Marketing|70000|   38|   Mid|
|     6|    Frank|       NULL|54000| NULL|   Low|
|     7|    Grace|Engineering|95000|   35|  High|
|     8|    Henry|         HR|58000|   45|   Low|
|     9|      Ivy|  Marketing|67000|   31|   Mid|
|    10|     Jack|Engineering|78000|   28|   Mid|
+------+---------+-----------+-----+-----+------+



In [43]:
df_1.orderBy(col("salary").desc()).show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  7|Grace|Engineering| 95000|  35|       High|
|  3|Carol|Engineering| 91000|  41|       High|
|  1|Alice|Engineering| 85000|  29|       High|
| 10| Jack|Engineering| 78000|  28|        Mid|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  8|Henry|         HR| 58000|  45|        Low|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  4|David|         HR|  NULL|  27|        Low|
+---+-----+-----------+------+----+-----------+



In [44]:
df_1.groupBy("department").agg(F.count("salary").alias("avg_salary")).show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|         4|
|         HR|         1|
|       NULL|         1|
|  Marketing|         3|
+-----------+----------+



In [45]:
df_2 = df_1.dropna()

In [46]:
df_2.show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  8|Henry|         HR| 58000| 45|        Low|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [47]:
df_2.write.csv("output/employes_clean", header = True, mode = "overwrite")